# 10. 통근 경로 결과 분석용 데이터 정리

- 누적 80% 전체 OD를 기준 테이블로 사용
- 외부 통근은 최종 TMAP·카카오 병합 경로를 결합
- 내부 통근은 원본 OD의 평균 이동시간·이동거리를 사용
- 분석에 필요한 변수만 정리하여 CSV로 저장


In [1]:
# =========================================================
# 0. 라이브러리
# =========================================================

from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display


In [2]:
# =========================================================
# 1. 프로젝트 경로 설정
# =========================================================

CURRENT_DIR = Path.cwd().resolve()

if CURRENT_DIR.name == "notebooks":
    REPO_DIR = CURRENT_DIR.parent
elif (CURRENT_DIR / "notebooks").exists():
    REPO_DIR = CURRENT_DIR
else:
    raise FileNotFoundError(
        "프로젝트 저장소 위치를 찾지 못했습니다.\n"
        f"현재 위치: {CURRENT_DIR}\n"
        "저장소 루트 또는 notebooks 폴더에서 실행하세요."
    )

WORKSPACE_DIR = REPO_DIR.parent
DATA_DIR = WORKSPACE_DIR / "project_data"
PROCESSED_DIR = DATA_DIR / "processed"
API_RESULT_DIR = DATA_DIR / "api_results"

# 누적 80% 전체 OD: 내부 + 외부 모두 포함
SELECTED_OD_FILE = (
    PROCESSED_DIR
    / "all_age_commute_od_selected_80.csv"
)

# 07번에서 별도 저장한 내부 OD: 검증용
INTERNAL_OD_FILE = (
    PROCESSED_DIR
    / "commute_internal_od_80.csv"
)

# TMAP 30,348건 + 카카오 보완 63건이 합쳐진 외부 경로 최종본
FINAL_ROUTE_FILE = (
    API_RESULT_DIR
    / "transit_routes_final_merged.csv"
)

OUTPUT_FILE = (
    PROCESSED_DIR
    / "commute_routes_analysis_ready.csv"
)

print("누적 80% 전체 OD:", SELECTED_OD_FILE)
print("내부 OD 검증 파일:", INTERNAL_OD_FILE)
print("외부 경로 최종본:", FINAL_ROUTE_FILE)
print("최종 저장 파일:", OUTPUT_FILE)


누적 80% 전체 OD: /Users/janghwayeong/Desktop/부트캠프_프로젝트/project_data/processed/all_age_commute_od_selected_80.csv
내부 OD 검증 파일: /Users/janghwayeong/Desktop/부트캠프_프로젝트/project_data/processed/commute_internal_od_80.csv
외부 경로 최종본: /Users/janghwayeong/Desktop/부트캠프_프로젝트/project_data/api_results/transit_routes_final_merged.csv
최종 저장 파일: /Users/janghwayeong/Desktop/부트캠프_프로젝트/project_data/processed/commute_routes_analysis_ready.csv


In [3]:
# =========================================================
# 2. CSV 읽기 및 공통 정리 함수
# =========================================================

def read_csv_clean(file_path: Path) -> pd.DataFrame:
    if not file_path.exists():
        raise FileNotFoundError(f"파일을 찾지 못했습니다: {file_path}")

    data = pd.read_csv(
        file_path,
        encoding="utf-8-sig",
        low_memory=False,
    )

    data.columns = (
        data.columns.astype(str)
        .str.replace("\ufeff", "", regex=False)
        .str.strip()
    )

    return data


def clean_code(series: pd.Series) -> pd.Series:
    return (
        series.astype("string")
        .str.strip()
        .str.replace(r"\.0$", "", regex=True)
    )


def prepare_od_key(
    data: pd.DataFrame,
    file_name: str,
) -> pd.DataFrame:
    data = data.copy()

    for column in ["거주동 코드", "근무동 코드"]:
        if column in data.columns:
            data[column] = clean_code(data[column])

    if "OD_KEY" in data.columns:
        data["OD_KEY"] = (
            data["OD_KEY"].astype("string").str.strip()
        )
    elif "OD_ID" in data.columns:
        data["OD_KEY"] = (
            data["OD_ID"].astype("string").str.strip()
        )
    elif {
        "거주동 코드",
        "근무동 코드",
    }.issubset(data.columns):
        data["OD_KEY"] = (
            data["거주동 코드"]
            + "_"
            + data["근무동 코드"]
        )
    else:
        raise KeyError(
            f"{file_name}에서 OD_KEY를 만들 수 없습니다.\n"
            f"현재 열: {data.columns.tolist()}"
        )

    return data


In [4]:
# =========================================================
# 3. 입력 파일 불러오기
# =========================================================

selected_df = prepare_od_key(
    read_csv_clean(SELECTED_OD_FILE),
    "누적 80% 전체 OD",
)

route_df = prepare_od_key(
    read_csv_clean(FINAL_ROUTE_FILE),
    "외부 경로 최종본",
)

if INTERNAL_OD_FILE.exists():
    internal_check_df = prepare_od_key(
        read_csv_clean(INTERNAL_OD_FILE),
        "내부 OD 검증 파일",
    )
else:
    internal_check_df = pd.DataFrame()
    print("주의: 내부 OD 검증 파일이 없어 전체 OD에서 직접 판별합니다.")

print(f"누적 80% 전체 OD: {len(selected_df):,}행")
print(f"외부 경로 최종본: {len(route_df):,}행")
print(f"외부 경로 고유 OD: {route_df['OD_KEY'].nunique():,}개")

if not internal_check_df.empty:
    print(f"내부 OD 검증 파일: {len(internal_check_df):,}행")


누적 80% 전체 OD: 30,839행
외부 경로 최종본: 30,411행
외부 경로 고유 OD: 30,411개
내부 OD 검증 파일: 428행


In [5]:
# =========================================================
# 4. 누적 80% OD 열 이름 표준화
# =========================================================

selected_df = selected_df.rename(
    columns={
        "출근 이동량": "출근_이동량",
        "최종 가중치": "최종_가중치",
        "목적지 출근비중": "목적지_출근비중",
        "누적 출근비중": "누적_출근비중",
    }
)

required_selected = [
    "OD_KEY",
    "거주동 코드",
    "거주동 이름",
    "근무동 코드",
    "근무동 이름",
    "출근_이동량",
    "거주동_전체_출근량",
    "목적지_출근비중",
    "누적_출근비중",
    "선택목적지_출근량합",
    "최종_가중치",
    "평균_이동시간_분",
    "평균_이동거리_m",
]

missing_selected = [
    column
    for column in required_selected
    if column not in selected_df.columns
]

if missing_selected:
    raise KeyError(
        "누적 80% OD 파일에 필요한 변수가 없습니다.\n"
        f"없는 변수: {missing_selected}\n"
        f"현재 변수: {selected_df.columns.tolist()}"
    )

selected_df["내부통근여부"] = (
    selected_df["거주동 코드"]
    .eq(selected_df["근무동 코드"])
)

print("전체 선택 OD:", f"{len(selected_df):,}")
print(
    "내부 통근 OD:",
    f"{selected_df['내부통근여부'].sum():,}",
)
print(
    "외부 통근 OD:",
    f"{(~selected_df['내부통근여부']).sum():,}",
)


전체 선택 OD: 30,839
내부 통근 OD: 428
외부 통근 OD: 30,411


In [6]:
# =========================================================
# 5. 내부 OD 분리 파일과 교차 검증
# =========================================================

internal_from_selected = set(
    selected_df.loc[
        selected_df["내부통근여부"],
        "OD_KEY",
    ].dropna()
)

if not internal_check_df.empty:
    internal_from_file = set(
        internal_check_df["OD_KEY"].dropna()
    )

    print(
        "전체 OD에서 판별한 내부 OD:",
        f"{len(internal_from_selected):,}",
    )
    print(
        "내부 OD 파일의 고유 OD:",
        f"{len(internal_from_file):,}",
    )
    print(
        "내부 파일에만 있는 OD:",
        f"{len(internal_from_file - internal_from_selected):,}",
    )
    print(
        "전체 OD에만 있는 내부 OD:",
        f"{len(internal_from_selected - internal_from_file):,}",
    )

    if internal_from_file != internal_from_selected:
        raise ValueError(
            "전체 OD에서 판별한 내부 OD와 "
            "commute_internal_od_80.csv가 일치하지 않습니다."
        )


전체 OD에서 판별한 내부 OD: 428
내부 OD 파일의 고유 OD: 428
내부 파일에만 있는 OD: 0
전체 OD에만 있는 내부 OD: 0


In [7]:
# =========================================================
# 6. 외부 경로 최종본 검증
# =========================================================

required_route = [
    "OD_KEY",
    "총소요시간_분",
    "총이동거리_m",
    "총이동거리_km",
    "예상대중교통요금_원",
    "총도보시간_분",
    "총도보거리_m",
    "환승횟수",
    "교통수단_순서",
    "최종경로유형",
    "최종데이터출처",
]

missing_route = [
    column
    for column in required_route
    if column not in route_df.columns
]

if missing_route:
    raise KeyError(
        "외부 경로 최종본에 필요한 변수가 없습니다.\n"
        f"없는 변수: {missing_route}\n"
        f"현재 변수: {route_df.columns.tolist()}"
    )

if route_df["OD_KEY"].duplicated().any():
    duplicate_keys = (
        route_df.loc[
            route_df["OD_KEY"].duplicated(keep=False),
            "OD_KEY",
        ]
        .drop_duplicates()
        .tolist()
    )
    raise ValueError(
        "외부 경로 최종본에 중복 OD_KEY가 있습니다.\n"
        f"중복 예시: {duplicate_keys[:10]}"
    )

external_key_set = set(
    selected_df.loc[
        ~selected_df["내부통근여부"],
        "OD_KEY",
    ].dropna()
)

route_key_set = set(
    route_df["OD_KEY"].dropna()
)

missing_external = external_key_set - route_key_set
extra_route = route_key_set - external_key_set

print("선택된 외부 OD:", f"{len(external_key_set):,}")
print("경로 결과 OD:", f"{len(route_key_set):,}")
print("경로 누락 외부 OD:", f"{len(missing_external):,}")
print("타겟 외 경로 OD:", f"{len(extra_route):,}")

if missing_external or extra_route:
    raise ValueError(
        "누적 80% 외부 OD와 최종 경로 파일의 OD_KEY가 일치하지 않습니다."
    )

print("\n[최종 데이터 출처]")
print(route_df["최종데이터출처"].value_counts(dropna=False))

print("\n[최종 경로 유형]")
print(route_df["최종경로유형"].value_counts(dropna=False))


선택된 외부 OD: 30,411
경로 결과 OD: 30,411
경로 누락 외부 OD: 0
타겟 외 경로 OD: 0

[최종 데이터 출처]
최종데이터출처
TMAP     30348
KAKAO       63
Name: count, dtype: int64

[최종 경로 유형]
최종경로유형
대중교통    30375
도보         36
Name: count, dtype: int64


In [8]:
# =========================================================
# 7. 외부 통근 분석용 경로 변수 선택
# =========================================================

optional_route_columns = [
    "버스_이용구간수",
    "지하철_이용구간수",
    "도보_구간수",
    "기차_이용구간수",
    "이용노선",
    "추천경로수",
    "전체추천경로수",
    "조회기준시각",
    "실제호출시각",
    "호출시각",
]

route_columns = (
    required_route
    + [
        column
        for column in optional_route_columns
        if column in route_df.columns
    ]
)

external_route = route_df[
    list(dict.fromkeys(route_columns))
].copy()

external_route["내부통근여부"] = False
external_route["경로값_산출방식"] = (
    external_route["최종데이터출처"]
    + "_최종경로"
)


In [9]:
# =========================================================
# 8. 전체 누적 80% OD와 외부 경로 결합
# =========================================================

analysis_od = selected_df.merge(
    external_route,
    on="OD_KEY",
    how="left",
    validate="one_to_one",
    suffixes=("", "_경로"),
)

# 병합 후 내부통근여부는 기준 테이블 값을 사용
if "내부통근여부_경로" in analysis_od.columns:
    analysis_od = analysis_od.drop(
        columns=["내부통근여부_경로"]
    )

print("병합 결과:", f"{len(analysis_od):,}행")
print("고유 OD:", f"{analysis_od['OD_KEY'].nunique():,}개")


병합 결과: 30,839행
고유 OD: 30,839개


In [10]:
# =========================================================
# 9. 내부 통근 경로값 적용
#
# 내부 통근은 동일 주민센터 좌표를 API에 넣지 않았으므로
# 원본 OD의 평균 이동시간·이동거리를 사용한다.
# 교통비는 근거가 없으므로 0원이 아니라 결측으로 둔다.
# =========================================================

internal_mask = analysis_od["내부통근여부"]

analysis_od.loc[
    internal_mask,
    "총소요시간_분",
] = analysis_od.loc[
    internal_mask,
    "평균_이동시간_분",
]

analysis_od.loc[
    internal_mask,
    "총이동거리_m",
] = analysis_od.loc[
    internal_mask,
    "평균_이동거리_m",
]

analysis_od.loc[
    internal_mask,
    "총이동거리_km",
] = (
    analysis_od.loc[
        internal_mask,
        "평균_이동거리_m",
    ]
    / 1000
)

analysis_od.loc[
    internal_mask,
    "예상대중교통요금_원",
] = np.nan

analysis_od.loc[
    internal_mask,
    "총도보시간_분",
] = np.nan

analysis_od.loc[
    internal_mask,
    "총도보거리_m",
] = np.nan

analysis_od.loc[
    internal_mask,
    "환승횟수",
] = np.nan

analysis_od.loc[
    internal_mask,
    "교통수단_순서",
] = pd.NA

analysis_od.loc[
    internal_mask,
    "최종경로유형",
] = "내부통근_원본OD"

analysis_od.loc[
    internal_mask,
    "최종데이터출처",
] = "원본_OD"

analysis_od.loc[
    internal_mask,
    "경로값_산출방식",
] = "원본_OD_내부통근"

print(
    "내부 통근 시간 결측:",
    analysis_od.loc[
        internal_mask,
        "총소요시간_분",
    ].isna().sum(),
)
print(
    "내부 통근 거리 결측:",
    analysis_od.loc[
        internal_mask,
        "총이동거리_m",
    ].isna().sum(),
)


내부 통근 시간 결측: 0
내부 통근 거리 결측: 0


In [11]:
# =========================================================
# 10. 분석용 파생변수 생성
# =========================================================

analysis_od["분석용_편도시간_분"] = pd.to_numeric(
    analysis_od["총소요시간_분"],
    errors="coerce",
)

analysis_od["분석용_편도거리_km"] = pd.to_numeric(
    analysis_od["총이동거리_km"],
    errors="coerce",
)

analysis_od["분석용_편도요금_원"] = pd.to_numeric(
    analysis_od["예상대중교통요금_원"],
    errors="coerce",
)

# 순수 도보 외부 경로는 요금 0원
pure_walk_mask = (
    (~analysis_od["내부통근여부"])
    & analysis_od["최종경로유형"].eq("도보")
)

analysis_od.loc[
    pure_walk_mask,
    "분석용_편도요금_원",
] = 0

analysis_od["도보시간비중"] = (
    pd.to_numeric(
        analysis_od["총도보시간_분"],
        errors="coerce",
    )
    / analysis_od["분석용_편도시간_분"]
)

analysis_od["도보시간비중"] = (
    analysis_od["도보시간비중"]
    .replace([np.inf, -np.inf], np.nan)
    .clip(lower=0, upper=1)
)

analysis_od["경로정보존재여부"] = (
    analysis_od["분석용_편도시간_분"].notna()
    & analysis_od["분석용_편도거리_km"].notna()
)

analysis_od["요금정보존재여부"] = (
    analysis_od["분석용_편도요금_원"].notna()
)


In [12]:
# =========================================================
# 11. 최종 저장 변수 선택
# =========================================================

final_columns = [
    "OD_KEY",
    "거주동 코드",
    "거주동 이름",
    "근무동 코드",
    "근무동 이름",
    "목적지_순위",
    "출근_이동량",
    "거주동_전체_출근량",
    "목적지_출근비중",
    "누적_출근비중",
    "선택목적지_출근량합",
    "최종_가중치",
    "평균_이동시간_분",
    "평균_이동거리_m",
    "내부통근여부",
    "분석용_편도시간_분",
    "분석용_편도거리_km",
    "분석용_편도요금_원",
    "총도보시간_분",
    "총도보거리_m",
    "도보시간비중",
    "환승횟수",
    "버스_이용구간수",
    "지하철_이용구간수",
    "도보_구간수",
    "기차_이용구간수",
    "교통수단_순서",
    "이용노선",
    "추천경로수",
    "전체추천경로수",
    "최종경로유형",
    "경로값_산출방식",
    "경로정보존재여부",
    "요금정보존재여부",
]

final_columns = [
    column
    for column in final_columns
    if column in analysis_od.columns
]

analysis_ready = (
    analysis_od[final_columns]
    .sort_values(
        [
            "거주동 코드",
            "출근_이동량",
            "근무동 코드",
        ],
        ascending=[True, False, True],
    )
    .reset_index(drop=True)
)

display(analysis_ready.head())
print("최종 분석용 OD:", f"{len(analysis_ready):,}행")


,OD_KEY,거주동 코드,거주동 이름,근무동 코드,근무동 이름,목적지_순위,출근_이동량,거주동_전체_출근량,목적지_출근비중,누적_출근비중,...,도보_구간수,기차_이용구간수,교통수단_순서,이용노선,추천경로수,전체추천경로수,최종경로유형,경로값_산출방식,경로정보존재여부,요금정보존재여부
0,11110515_11110530,11110515,청운효자동,11110530,사직동,1,55710.25,564412.04,0.098705,0.098705,...,2.0,0.0,WALK → BUS → WALK,지선:1711,4.0,4.0,대중교통,TMAP_최종경로,True,True
1,11110515_11110615,11110515,청운효자동,11110615,종로1.2.3.4가동,2,47373.52,564412.04,0.083934,0.182639,...,3.0,0.0,WALK → BUS → WALK → BUS → WALK,지선:1020 → 간선:272,10.0,10.0,대중교통,TMAP_최종경로,True,True
2,11110515_11110515,11110515,청운효자동,11110515,청운효자동,3,43228.92,564412.04,0.076591,0.259230,...,NaN,NaN,NaN,NaN,NaN,NaN,내부통근_원본OD,원본_OD_내부통근,True,False
3,11110515_11140550,11110515,청운효자동,11140550,명동,4,21721.17,564412.04,0.038485,0.297715,...,3.0,0.0,WALK → BUS → WALK → BUS → WALK,지선:7022 → 순환:TOUR11,10.0,10.0,대중교통,TMAP_최종경로,True,True
4,11110515_11140520,11110515,청운효자동,11140520,소공동,5,17247.38,564412.04,0.030558,0.328273,...,2.0,0.0,WALK → BUS → WALK,지선:1711,10.0,10.0,대중교통,TMAP_최종경로,True,True


최종 분석용 OD: 30,839행


In [13]:
# =========================================================
# 12. 최종 품질검사
# =========================================================

if analysis_ready["OD_KEY"].duplicated().any():
    raise ValueError("최종 데이터에 중복 OD_KEY가 있습니다.")

if len(analysis_ready) != len(selected_df):
    raise ValueError(
        "최종 데이터 행 수가 누적 80% 전체 OD와 다릅니다."
    )

print(
    "경로시간 결측:",
    f"{analysis_ready['분석용_편도시간_분'].isna().sum():,}",
)
print(
    "경로거리 결측:",
    f"{analysis_ready['분석용_편도거리_km'].isna().sum():,}",
)
print(
    "요금 결측:",
    f"{analysis_ready['분석용_편도요금_원'].isna().sum():,}",
)

# 요금 결측 원인 분해
fee_missing_summary = (
    analysis_od.loc[
        analysis_od["분석용_편도요금_원"].isna(),
        [
            "내부통근여부",
            "최종경로유형",
            "최종데이터출처",
        ],
    ]
    .value_counts(dropna=False)
    .rename("OD수")
    .reset_index()
)

print("\n[요금 결측 원인]")
display(fee_missing_summary)

internal_count = int(analysis_ready["내부통근여부"].sum())
print("내부 통근 OD 수:", f"{internal_count:,}")

weight_check = (
    analysis_ready
    .groupby(
        ["거주동 코드", "거주동 이름"],
        observed=True,
    )["최종_가중치"]
    .sum()
)

weight_deviation = (weight_check - 1.0).abs()

print(
    "거주동별 최종 가중치 합 범위:",
    weight_check.min(),
    "~",
    weight_check.max(),
)
print(
    "1에서 가장 크게 벗어난 정도:",
    weight_deviation.max(),
)

if not np.allclose(
    weight_check.to_numpy(dtype=float),
    1.0,
    atol=1e-6,
):
    raise ValueError(
        "일부 거주동의 최종 가중치 합이 허용 오차를 벗어났습니다."
    )

print("가중치 합 검증 완료: 부동소수점 허용 오차 안에서 모두 1입니다.")


경로시간 결측: 0
경로거리 결측: 0
요금 결측: 433

[요금 결측 원인]


,내부통근여부,최종경로유형,최종데이터출처,OD수
0,True,내부통근_원본OD,원본_OD,428
1,False,대중교통,KAKAO,5


내부 통근 OD 수: 428
거주동별 최종 가중치 합 범위: 0.99999993 ~ 1.00000007
1에서 가장 크게 벗어난 정도: 7.000000001866624e-08
가중치 합 검증 완료: 부동소수점 허용 오차 안에서 모두 1입니다.


In [18]:
# =========================================================
# 카카오 대중교통 선택 요금 누락 보완
# 선택 경로 요금이 없고 최소요금만 있으면 최소요금 사용
# =========================================================

kakao_min_fare_map = (
    route_df[
        [
            "OD_KEY",
            "카카오_대중교통_최소요금_원",
        ]
    ]
    .drop_duplicates("OD_KEY")
    .set_index("OD_KEY")[
        "카카오_대중교통_최소요금_원"
    ]
)

kakao_fee_missing_mask = (
    (~analysis_od["내부통근여부"])
    & analysis_od["최종경로유형"].eq("대중교통")
    & analysis_od["분석용_편도요금_원"].isna()
)

analysis_od.loc[
    kakao_fee_missing_mask,
    "분석용_편도요금_원",
] = pd.to_numeric(
    analysis_od.loc[
        kakao_fee_missing_mask,
        "OD_KEY",
    ].map(kakao_min_fare_map),
    errors="coerce",
)

In [19]:
analysis_od["요금산출방식"] = "API_선택경로요금"

analysis_od.loc[
    analysis_od["내부통근여부"],
    "요금산출방식",
] = "내부통근_요금미산출"

analysis_od.loc[
    kakao_fee_missing_mask
    & analysis_od["분석용_편도요금_원"].notna(),
    "요금산출방식",
] = "카카오_최소요금_보완"

In [20]:
pure_walk_mask = (
    (~analysis_od["내부통근여부"])
    & analysis_od["최종경로유형"].eq("도보")
)

analysis_od.loc[
    pure_walk_mask,
    "분석용_편도요금_원",
] = 0

analysis_od.loc[
    pure_walk_mask,
    "요금산출방식",
] = "순수도보_0원"

In [21]:
print(
    "전체 요금 결측:",
    analysis_od["분석용_편도요금_원"].isna().sum(),
)

print(
    "외부 통근 요금 결측:",
    (
        (~analysis_od["내부통근여부"])
        & analysis_od["분석용_편도요금_원"].isna()
    ).sum(),
)

print(
    "내부 통근 요금 결측:",
    (
        analysis_od["내부통근여부"]
        & analysis_od["분석용_편도요금_원"].isna()
    ).sum(),
)

전체 요금 결측: 428
외부 통근 요금 결측: 0
내부 통근 요금 결측: 428


In [22]:
# =========================================================
# 13. 최종 분석용 CSV 생성 및 저장
# =========================================================

# 저장할 최종 변수
final_columns = [
    "OD_KEY",
    "거주동 코드",
    "거주동 이름",
    "근무동 코드",
    "근무동 이름",
    "목적지_순위",
    "출근_이동량",
    "거주동_전체_출근량",
    "목적지_출근비중",
    "누적_출근비중",
    "선택목적지_출근량합",
    "최종_가중치",
    "평균_이동시간_분",
    "평균_이동거리_m",
    "내부통근여부",
    "분석용_편도시간_분",
    "분석용_편도거리_km",
    "분석용_편도요금_원",
    "요금산출방식",
    "총도보시간_분",
    "총도보거리_m",
    "도보시간비중",
    "환승횟수",
    "버스_이용구간수",
    "지하철_이용구간수",
    "도보_구간수",
    "기차_이용구간수",
    "교통수단_순서",
    "이용노선",
    "추천경로수",
    "전체추천경로수",
    "최종경로유형",
    "경로값_산출방식",
    "경로정보존재여부",
    "요금정보존재여부",
]

# 실제 존재하는 열만 선택
final_columns = [
    column
    for column in final_columns
    if column in analysis_od.columns
]

# 카카오 요금 보정 이후의 analysis_od로 최종 데이터 재생성
analysis_ready = (
    analysis_od[final_columns]
    .sort_values(
        [
            "거주동 코드",
            "출근_이동량",
            "근무동 코드",
        ],
        ascending=[True, False, True],
    )
    .reset_index(drop=True)
)

# 요금정보 존재 여부도 보정 결과 기준으로 다시 계산
analysis_ready["요금정보존재여부"] = (
    analysis_ready["분석용_편도요금_원"].notna()
)

# ---------------------------------------------------------
# 최종 검사
# ---------------------------------------------------------

if analysis_ready["OD_KEY"].duplicated().any():
    raise ValueError("최종 데이터에 중복 OD_KEY가 있습니다.")

if len(analysis_ready) != len(selected_df):
    raise ValueError(
        "최종 데이터 행 수가 누적 80% 전체 OD와 다릅니다.\n"
        f"전체 OD: {len(selected_df):,}\n"
        f"최종 데이터: {len(analysis_ready):,}"
    )

external_fee_missing = (
    (~analysis_ready["내부통근여부"])
    & analysis_ready["분석용_편도요금_원"].isna()
).sum()

internal_fee_missing = (
    analysis_ready["내부통근여부"]
    & analysis_ready["분석용_편도요금_원"].isna()
).sum()

print("최종 행 수:", f"{len(analysis_ready):,}")
print(
    "경로시간 결측:",
    f"{analysis_ready['분석용_편도시간_분'].isna().sum():,}",
)
print(
    "경로거리 결측:",
    f"{analysis_ready['분석용_편도거리_km'].isna().sum():,}",
)
print(
    "외부 통근 요금 결측:",
    f"{external_fee_missing:,}",
)
print(
    "내부 통근 요금 결측:",
    f"{internal_fee_missing:,}",
)

if external_fee_missing > 0:
    raise ValueError(
        "외부 통근에 요금 결측이 남아 있습니다. "
        "카카오 최소요금 보정 셀을 먼저 확인하세요."
    )

# ---------------------------------------------------------
# CSV 저장
# ---------------------------------------------------------

OUTPUT_FILE.parent.mkdir(
    parents=True,
    exist_ok=True,
)

analysis_ready.to_csv(
    OUTPUT_FILE,
    index=False,
    encoding="utf-8-sig",
)

print("\n저장 완료:", OUTPUT_FILE)
print("저장 행 수:", f"{len(analysis_ready):,}")
print("저장 열 수:", f"{len(analysis_ready.columns):,}")

display(analysis_ready.head())

최종 행 수: 30,839
경로시간 결측: 0
경로거리 결측: 0
외부 통근 요금 결측: 0
내부 통근 요금 결측: 428

저장 완료: /Users/janghwayeong/Desktop/부트캠프_프로젝트/project_data/processed/commute_routes_analysis_ready.csv
저장 행 수: 30,839
저장 열 수: 35


,OD_KEY,거주동 코드,거주동 이름,근무동 코드,근무동 이름,목적지_순위,출근_이동량,거주동_전체_출근량,목적지_출근비중,누적_출근비중,...,도보_구간수,기차_이용구간수,교통수단_순서,이용노선,추천경로수,전체추천경로수,최종경로유형,경로값_산출방식,경로정보존재여부,요금정보존재여부
0,11110515_11110530,11110515,청운효자동,11110530,사직동,1,55710.25,564412.04,0.098705,0.098705,...,2.0,0.0,WALK → BUS → WALK,지선:1711,4.0,4.0,대중교통,TMAP_최종경로,True,True
1,11110515_11110615,11110515,청운효자동,11110615,종로1.2.3.4가동,2,47373.52,564412.04,0.083934,0.182639,...,3.0,0.0,WALK → BUS → WALK → BUS → WALK,지선:1020 → 간선:272,10.0,10.0,대중교통,TMAP_최종경로,True,True
2,11110515_11110515,11110515,청운효자동,11110515,청운효자동,3,43228.92,564412.04,0.076591,0.259230,...,NaN,NaN,NaN,NaN,NaN,NaN,내부통근_원본OD,원본_OD_내부통근,True,False
3,11110515_11140550,11110515,청운효자동,11140550,명동,4,21721.17,564412.04,0.038485,0.297715,...,3.0,0.0,WALK → BUS → WALK → BUS → WALK,지선:7022 → 순환:TOUR11,10.0,10.0,대중교통,TMAP_최종경로,True,True
4,11110515_11140520,11110515,청운효자동,11140520,소공동,5,17247.38,564412.04,0.030558,0.328273,...,2.0,0.0,WALK → BUS → WALK,지선:1711,10.0,10.0,대중교통,TMAP_최종경로,True,True
